# Búsqueda Semántica con Embeddings y RAG

> Aprende a construir un sistema de búsqueda semántica usando embeddings de sentence-transformers
> y ChromaDB, e integra un chatbot RAG que responde con contexto recuperado.

## Objetivos
- Indexar documentos con embeddings en ChromaDB
- Comparar búsqueda por keywords vs búsqueda semántica
- Construir un chatbot RAG con Claude y ChromaDB
- Evaluar la relevancia de los resultados recuperados

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic sentence-transformers chromadb --quiet

## 2. Configuración e inicialización

In [ ]:
import anthropic
import chromadb
from sentence_transformers import SentenceTransformer

# Cliente Anthropic — API key desde variable de entorno
cliente_ia = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"

# Modelo de embeddings multilingüe
print("Cargando modelo de embeddings...")
modelo_embeddings = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Base de datos vectorial en memoria (sin persistencia, ideal para demos)
chroma = chromadb.EphemeralClient()
coleccion = chroma.create_collection("documentos_ia")

print("Sistema de búsqueda semántica inicializado.")

## 3. Indexar documentos de ejemplo

In [ ]:
# Corpus de documentos sobre IA y tecnología
documentos = [
    {"id": "doc1", "texto": "El aprendizaje automático permite a las máquinas aprender patrones de datos sin ser programadas explícitamente.", "categoria": "ml"},
    {"id": "doc2", "texto": "Las redes neuronales artificiales están inspiradas en el funcionamiento del cerebro humano.", "categoria": "deep_learning"},
    {"id": "doc3", "texto": "El procesamiento del lenguaje natural (NLP) permite a los ordenadores entender y generar texto humano.", "categoria": "nlp"},
    {"id": "doc4", "texto": "Los transformers revolucionaron el NLP gracias al mecanismo de atención que captura relaciones entre tokens.", "categoria": "nlp"},
    {"id": "doc5", "texto": "El fine-tuning adapta un modelo preentrenado a una tarea específica con un dataset más pequeño.", "categoria": "ml"},
    {"id": "doc6", "texto": "RAG combina recuperación de información con generación de texto para respuestas más precisas y actualizadas.", "categoria": "llm"},
    {"id": "doc7", "texto": "Los embeddings son representaciones vectoriales de texto que capturan su significado semántico.", "categoria": "nlp"},
    {"id": "doc8", "texto": "El prompt engineering es el arte de diseñar instrucciones efectivas para modelos de lenguaje.", "categoria": "llm"},
    {"id": "doc9", "texto": "La IA generativa puede crear texto, imágenes, audio y video a partir de instrucciones en lenguaje natural.", "categoria": "generativa"},
    {"id": "doc10", "texto": "Los agentes de IA pueden ejecutar tareas complejas usando herramientas externas y planificación.", "categoria": "agentes"},
]

# Generar embeddings e indexar
textos = [d["texto"] for d in documentos]
embeddings = modelo_embeddings.encode(textos).tolist()

coleccion.add(
    ids=[d["id"] for d in documentos],
    documents=textos,
    embeddings=embeddings,
    metadatas=[{"categoria": d["categoria"]} for d in documentos]
)

print(f"Indexados {len(documentos)} documentos en ChromaDB.")

## 4. Búsqueda semántica vs búsqueda por keywords

In [ ]:
def busqueda_semantica(query: str, n_resultados: int = 3) -> list[dict]:
    """Busca documentos similares por significado semántico."""
    embedding_query = modelo_embeddings.encode([query]).tolist()
    resultados = coleccion.query(
        query_embeddings=embedding_query,
        n_results=n_resultados
    )
    return [
        {"texto": doc, "distancia": dist}
        for doc, dist in zip(resultados["documents"][0], resultados["distances"][0])
    ]

def busqueda_keywords(query: str, documentos: list[dict]) -> list[dict]:
    """Búsqueda simple por palabras clave (sin semántica)."""
    palabras = query.lower().split()
    coincidencias = []
    for doc in documentos:
        score = sum(1 for p in palabras if p in doc["texto"].lower())
        if score > 0:
            coincidencias.append({"texto": doc["texto"], "score": score})
    return sorted(coincidencias, key=lambda x: x["score"], reverse=True)[:3]

# Consulta que ilustra la diferencia: no contiene las palabras exactas del documento
QUERY = "¿Cómo pueden las máquinas comprender el lenguaje de los seres humanos?"

print(f"Query: '{QUERY}'\n")

print("=== BÚSQUEDA POR KEYWORDS ===")
kw = busqueda_keywords(QUERY, documentos)
for i, r in enumerate(kw, 1):
    print(f"{i}. (score={r['score']}) {r['texto'][:80]}...")

print("\n=== BÚSQUEDA SEMÁNTICA ===")
sem = busqueda_semantica(QUERY)
for i, r in enumerate(sem, 1):
    print(f"{i}. (dist={r['distancia']:.3f}) {r['texto'][:80]}...")

## 5. Chatbot RAG con Claude

El chatbot recupera contexto relevante antes de responder, garantizando respuestas fundamentadas.

In [ ]:
def chatbot_rag(pregunta: str, n_contextos: int = 3) -> dict:
    """Responde preguntas usando RAG: recupera contexto relevante y genera respuesta con Claude."""
    # Paso 1: recuperar documentos relevantes
    contextos = busqueda_semantica(pregunta, n_contextos)
    contexto_texto = "\n".join([f"- {c['texto']}" for c in contextos])

    # Paso 2: generar respuesta con contexto
    prompt = f"""Responde la siguiente pregunta basándote ÚNICAMENTE en el contexto proporcionado.
Si la respuesta no está en el contexto, dilo explícitamente.

Contexto:
{contexto_texto}

Pregunta: {pregunta}"""

    respuesta = cliente_ia.messages.create(
        model=MODELO,
        max_tokens=512,
        messages=[{"role": "user", "content": prompt}]
    )

    return {
        "respuesta": respuesta.content[0].text,
        "contextos_usados": contextos
    }

# Prueba del chatbot RAG
preguntas = [
    "¿Qué son los embeddings y para qué sirven?",
    "¿Cómo funciona RAG?",
    "¿Qué es el quantum computing?"  # Fuera del contexto disponible
]

for pregunta in preguntas:
    print(f"\n{'='*60}")
    print(f"Pregunta: {pregunta}")
    resultado = chatbot_rag(pregunta)
    print(f"Respuesta: {resultado['respuesta']}")
    print(f"Contextos usados: {len(resultado['contextos_usados'])}")